# Recherche évolutionnaire interactive

Lance l'algorithme et affiche en direct les **10 meilleures molécules** après chaque génération.

In [1]:
import os, sys
from pathlib import Path

# racine du projet = premier dossier parent qui contient evomol/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "evomol").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # les filtres ChEMBL/ZINC lisent external_data/ en relatif

from IPython.display import display, clear_output
import ipywidgets as widgets
from rdkit.Chem import Draw, MolFromSmiles

from evomol import evaluation as ev
from evomol.representation import MolecularGraph
from evomol.search.parameters import search_parameters

print("Imports OK")

Imports OK


## Paramètres de l'expérience

In [5]:
# ── Molécule de départ et espace chimique ───────────────────────────────────
SMILES_DEPART = "c1ccccc1"          # benzène (possède des électrons π)
#   SMILES quelconque : molécule d'où part la recherche (doit passer les filtres).
ACCEPTED_ATOMS  = ["C", "N", "O", "F", "S"]
#   éléments autorisés dans les molécules générées.
MAX_HEAVY_ATOMS = 38
#   entier >= 1 : nb max d'atomes lourds (hors hydrogènes).
UNIQUE_KEKULE   = False
#   False : SMILES kékulé tel que produit par la mutation (une molécule aromatique peut avoir
#           deux SMILES ; défaut, le plus rapide).
#   True  : un seul SMILES par molécule (plus de doublons, ~15–30 % de candidats/s en moins).

# ── Critère d'optimisation ──────────────────────────────────────────────────
# Choisir parmi : "huckel_homo_lambda", "huckel_lumo_lambda", "huckel_gap_lambda"
FITNESS = "huckel_homo_lambda"

# ── Arrêt de la recherche ───────────────────────────────────────────────────
MAX_GENERATIONS = 10_000
MAX_TIME_SEC    = 400       # durée max en secondes

# ── Population et sélection ─────────────────────────────────────────────────
POPULATION_SIZE = 100
#   entier >= 1 : nb de molécules gardées. Si > 1, la population part de SMILES_DEPART
#   seule et se remplit au fil des générations (cases vides remplies en premier).
NB_REPLACEMENTS = 10
#   entier, 1 <= NB_REPLACEMENTS <= POPULATION_SIZE : nb de pires molécules remplacées par génération.
SELECTION       = "best"
#   ordre dans lequel les molécules sont choisies pour être mutées :
#   "best"     : de la meilleure à la pire.
#   "random"   : ordre aléatoire.
#   "roulette" : tirage pondéré par la fitness. ATTENTION : suppose toutes les fitness >= 0
#                (avec des "min"/"target", fitness négatives → tirage faussé).
SEED            = 42
#   entier : recherche reproductible (même entier -> même recherche).
#   None   : recherche différente à chaque exécution.

# ── Mutation ────────────────────────────────────────────────────────────────
DEPTH                = 3
#   entier >= 1 : nb (max) d'actions appliquées à la suite dans une mutation.
RANDOM_DEPTH         = True
#   True  : nb d'actions tiré entre 1 et DEPTH à chaque mutation.
#   False : exactement DEPTH actions.
NB_MAX_TRIES         = 50
#   entier >= 1 : nb de mutants essayés par parent avant de passer au parent suivant.
UNIFORM_OVER_ACTIONS = True
#   False : on tire d'abord un type d'action, puis une action de ce type (types équiprobables).
#   True  : tirage uniforme sur toutes les actions possibles (types avec beaucoup
#           d'actions, ex. ChangeBondMG / MoveGroupMG, tirés plus souvent). Ignore PRESELECT_ACTIONS.
PRESELECT_ACTIONS    = True
#   True  : ne liste que les actions du type tiré (rapide).
#   False : liste toutes les actions avant de tirer (même loi, plus lent).
REMEMBER_INVALID     = True
#   True  : les SMILES déjà jugés invalides ne sont pas réévalués (même recherche, plus rapide).
#   False : chaque candidat est réévalué.

# ── Espace d'action ─────────────────────────────────────────────────────────
INSERT_CARBON       = True
#   True  : InsertCarbonMG disponible. Le C inséré est un sp3 non terminal :
#           il est toujours rejeté par le filtre Hückel.
#   False : InsertCarbonMG retiré.
AVOID_SP3_ON_SP3    = True
#   True  : AddAtomMG ne greffe pas de C sp3 sur un C sp3 (molécules que Hückel rejetterait).
#   False : pas de restriction.
ADD_ATOM_MULTIPLE_BONDS = True
#   True  : AddAtomMG peut aussi greffer le nouvel atome par une double ou triple liaison
#           (=CH2, =O, #N…) quand les valences le permettent. Le nouvel atome n'est alors
#           pas sp3 : permet d'allonger le système π même quand MAX_SP3 est atteint.
#   False : liaison simple uniquement.
AVOID_BOND_BREAKING = False
#   True  : ChangeBondMG ne supprime jamais de liaison (les cycles ne s'ouvrent plus par ChangeBondMG).
#   False : les liaisons de cycle peuvent être ouvertes.
AVOID_BOND_FORMING  = False
#   True  : ChangeBondMG ne crée jamais de liaison entre deux atomes non liés (les cycles ne se ferment plus).
#   False : pas de restriction.
WITH_ADD_GROUP      = False
#   True  : ajoute AddGroupMG (greffe thiophène, benzène, nitro, azide, acide sulfonique).
WITH_REMOVE_GROUP   = False
#   True  : ajoute RemoveGroupMG (retire un groupe d'atomes).

# ── Filtres ── appliqués dans cet ordre : Hückel → sp3 → ChEMBL/ZINC → Hetero
FILTER_HUCKEL  = True
#   True / False : compatibilité Hückel (HuckelCompatible) : tout C sp3 doit être en bout de chaîne.
MAX_SP3        = None
#   None        : filtre inactif.
#   entier >= 0 : nb max d'atomes sans liaison π, tous éléments confondus (FilterSP3).
FILTER_REALISM = "chembl_zinc"
#   "chembl_zinc" : filtres GCF + ECFP sur la base ChEMBL + ZINC (la plus large).
#   "chembl"      : filtres GCF + ECFP sur la base ChEMBL seule (plus restrictif).
#   None          : aucun filtre de réalisme (ni GCF ni ECFP).
MAX_RING_N     = 3
#   entier >= 0 : nb max d'azotes en position de cycle (3 : triazoles OK, tétrazoles rejetés).
#   Ignoré si FILTER_HETERO = False.
FILTER_HETERO  = False
#   True / False : chaînes d'hétéroatomes (HeteroatomChainFilter) : pas 3 hétéroatomes
#   consécutifs hors cycle, pas de O-O-O, au plus MAX_RING_N azotes en cycle.

# ── Orbitales observées ── sélecteur : "HOMO", "LUMO", "HOMO-1", "LUMO+2", … ou indice entier
ORBITAL_LAMBDA          = "HOMO-4"
#   orbitale dont on lit λ.
LAMBDA_OVERLAP_ORBITALS = ("HOMO", "LUMO")
#   2 orbitales ou plus : recouvrement spatial Σᵢ ∏ⱼ |cⱼᵢ| (ignore le signe).

# Noms des propriétés orbitalaires, d'après les orbitales choisies ci-dessus
ORBITAL_LAMBDA_NAME = ev.HuckelOrbitalLambda(ORBITAL_LAMBDA).name                       # ex. "huckel_orbital_lambda_HOMO-1"
LAMBDA_OVERLAP_NAME = ev.HuckelOrbitalLambdaOverlap(list(LAMBDA_OVERLAP_ORBITALS)).name # ex. "huckel_lambda_HOMO_LUMO"

# ── Objectif de fitness ───────────────────────────────────────────────────────
# ("nom_evaluateur", "max"|"min"|"target", poids, cible_ou_None)
#   "max"    : valeur la plus grande possible  (cible = None)
#   "min"    : valeur la plus petite possible  (cible = None)
#   "target" : valeur la plus proche de cible
# Toutes les propriétés calculées par evolutionary_algorithm.py sont listées ;
# décommenter une ligne pour l'utiliser.
fitness_goals = [
    # ("huckel_gap_lambda",  "max",    1.0, None),  # gap HOMO-LUMO le plus petit possible
    ("huckel_homo_lambda", "min", 1.0, None),   # HOMO λ proche de 0 = plus haute 
    # ("huckel_lumo_lambda", "target", 1.0, 0.2),   # lumo λ proche de 0 = plus base 
    # (LAMBDA_OVERLAP_NAME, "min", 3.0, None),  # maximiser l'overlap densité HOMO×LUMO
    # (ORBITAL_LAMBDA_NAME, "min", 1.0, None),  # λ de l'orbitale ORBITAL_LAMBDA
    # ── Autres propriétés (hors Hückel) ──
    # ("n_heavy_atoms",     "target", 1.0, 20),   # nb d'atomes lourds proche de 20
    # ("mol_weight",        "max",    1.0, None), # masse molaire (g/mol)
    ("SAScore",           "min",    1.0, None), # SA score : 1 (facile) → 10 (difficile à synthétiser)
    # ("NormalizedSAScore", "max",    1.0, None), # SA score normalisé : 1 (facile) → 0 (difficile)
    # ("zinc_normalized_sa_score", "max", 1.0, None), # SA score normalisé ZINC (grand = facile)
    # ("QED",               "max",    1.0, None), # drug-likeness : 0 → 1
    # ("logP",              "max",    1.0, None), # lipophilie
    # ("zinc_normalized_logP", "max", 1.0, None), # logP normalisé ZINC
    # ("CycleScore",        "max",    1.0, None), # 0 si cycles ≤ 6, négatif sinon
    # ("zinc_normalized_cycle_score", "max", 1.0, None), # score de cycle normalisé ZINC
    # ("PLogP",             "max",    1.0, None), # logP pénalisé (logP + SA + cycles, normalisés ZINC)
]

# ── Sauvegarde ──────────────────────────────────────────────────────────────
OUTPUT_DIR            = None
#   None : output/huckel_homo/ à la racine du projet ; sinon chemin d'un dossier.
SAVE_N_GENERATIONS    = 50
#   entier >= 0 : sauver la population toutes les N générations (0 : jamais).
SAVE_FINAL_POPULATION = True
#   True / False : sauver la population finale.

# ── Affichage ───────────────────────────────────────────────────────────────
TOP_N           = 30        # nombre de meilleures molécules à afficher
AFFICHER_TOUTES = 50        # afficher toutes les N générations (1 = chaque génération)

print(f"Fitness : {FITNESS}")
print(f"Départ  : {SMILES_DEPART}")
print(f"Durée   : {MAX_TIME_SEC}s  max {MAX_GENERATIONS} générations")

Fitness : huckel_homo_lambda
Départ  : c1ccccc1
Durée   : 400s  max 10000 générations


## Callback de visualisation

In [6]:
def _goal_details(mol):
    """Retourne [(nom, valeur)] pour chaque objectif de fitness_goals."""
    parts = []
    for name, *_ in fitness_goals:
        try:
            parts.append((name, mol.value(name)))
        except Exception:
            pass
    return parts

def afficher_top(population, generation):
    """Met à jour best_seen et affiche les TOP_N meilleures molécules jamais vues."""
    for mol in population:
        smiles  = mol.get_representation(MolecularGraph).canonical_smiles
        fitness = mol.value(search_parameters.fitness_criteria)
        if smiles not in best_seen or fitness > best_seen[smiles][0]:
            best_seen[smiles] = (fitness, mol)

    top_items = sorted(best_seen.values(), key=lambda x: x[0], reverse=True)[:TOP_N]

    mols, legends = [], []
    for rank, (fitness, mol) in enumerate(top_items, start=1):
        smiles = mol.get_representation(MolecularGraph).canonical_smiles
        rdmol  = MolFromSmiles(smiles)
        if rdmol is None:
            continue
        mols.append(rdmol)
        legend = f"#{rank}  fitness={fitness:+.4f}"
        for name, val in _goal_details(mol):
            legend += f"\n{name}={val:+.4f}"
        legend += f"\n{smiles}"
        legends.append(legend)

    if not mols:
        return

    n_cols = min(5, len(mols))
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=n_cols,
        subImgSize=(300, 260),
        legends=legends,
        returnPNG=False,
    )

    best_fitness, best_mol = top_items[0]
    detail_str = "  ".join(f"{n}={v:+.4f}" for n, v in _goal_details(best_mol))
    header = f"Génération {generation:5d}  │  Meilleure fitness : {best_fitness:+.4f}  │  {detail_str}"

    # Mise à jour sur place dans la sortie de la cellule de lancement
    # (pas de widget Output → évite les vues multiples sous VS Code)
    clear_output(wait=True)
    print(header)
    display(img)

print("Callback défini.")

Callback défini.


## Lancement de la recherche

In [7]:
import importlib.util

# Charger le script d'optimisation
_spec = importlib.util.spec_from_file_location(
    "evolutionary_algorithm",
    ROOT / "scripts" / "Lea" / "evolutionary_algorithm.py",
)
_ea = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ea)

# Réinitialiser l'état à chaque run (évite le double affichage)
best_seen = {}
out = widgets.Output()
display(out)

print("Démarrage…")
evo_mol = _ea.main(
    start_smiles=SMILES_DEPART,
    accepted_atoms=ACCEPTED_ATOMS,
    max_heavy_atoms=MAX_HEAVY_ATOMS,
    unique_kekule=UNIQUE_KEKULE,
    huckel=FILTER_HUCKEL,
    max_sp3=MAX_SP3,
    realism=FILTER_REALISM,
    hetero=FILTER_HETERO,
    max_ring_n=MAX_RING_N,
    orbital_lambda=ORBITAL_LAMBDA,
    lambda_overlap_orbitals=LAMBDA_OVERLAP_ORBITALS,
    fitness_goals=fitness_goals,
    max_generations=MAX_GENERATIONS,
    max_time=MAX_TIME_SEC,
    population_size=POPULATION_SIZE,
    nb_replacements=NB_REPLACEMENTS,
    selection=SELECTION,
    depth=DEPTH,
    random_depth=RANDOM_DEPTH,
    nb_max_tries=NB_MAX_TRIES,
    remember_invalid=REMEMBER_INVALID,
    preselect_actions=PRESELECT_ACTIONS,
    uniform_over_actions=UNIFORM_OVER_ACTIONS,
    insert_carbon=INSERT_CARBON,
    avoid_sp3_on_sp3=AVOID_SP3_ON_SP3,
    add_atom_multiple_bonds=ADD_ATOM_MULTIPLE_BONDS,
    avoid_bond_breaking=AVOID_BOND_BREAKING,
    avoid_bond_forming=AVOID_BOND_FORMING,
    with_add_group=WITH_ADD_GROUP,
    with_remove_group=WITH_REMOVE_GROUP,
    random_seed=SEED,
    generation_callback=afficher_top,
    callback_every_n=AFFICHER_TOUTES,
    output_dir=OUTPUT_DIR,
    save_n_generations=SAVE_N_GENERATIONS,
    save_final_population=SAVE_FINAL_POPULATION,
)
print("\nRecherche terminée.")

Output()

Démarrage…
INFO: Init the evolutionary algorithm.
INFO: Start the evolutionary algorithm.
INFO: Generation      1 - Mean: -4.2841 - Max: -3.5221 - Min: -4.9807 - C1=CC=CC=C1 - huckel_homo_lambda: 1.0000 - SAScore: 2.5221
INFO: Generation      2 - Mean: -4.4001 - Max: -3.5221 - Min: -5.1964 - C1=CC=CC=C1 - huckel_homo_lambda: 1.0000 - SAScore: 2.5221
INFO: Generation      3 - Mean: -4.3921 - Max: -3.4420 - Min: -5.1964 - O=C1C=CC=C1 - huckel_homo_lambda: 0.6180 - SAScore: 2.8239
INFO: Generation      4 - Mean: -4.4448 - Max: -3.4420 - Min: -5.1979 - O=C1C=CC=C1 - huckel_homo_lambda: 0.6180 - SAScore: 2.8239
INFO: Generation      5 - Mean: -4.3460 - Max: -3.0140 - Min: -5.1979 - CC(C)=O - huckel_homo_lambda: 1.3591 - SAScore: 1.6549
INFO: Generation      6 - Mean: -4.2527 - Max: -2.8905 - Min: -5.1979 - CC(=O)O - huckel_homo_lambda: 1.3756 - SAScore: 1.5149
INFO: Generation      7 - Mean: -4.1952 - Max: -2.8905 - Min: -5.1979 - CC(=O)O - huckel_homo_lambda: 1.3756 - SAScore: 1.5149
INFO:

## Résultats finaux

In [ ]:
# Ajouter les molécules finales (générations non couvertes par le callback)
for mol in evo_mol.population:
    smiles  = mol.get_representation(MolecularGraph).canonical_smiles
    fitness = mol.value(search_parameters.fitness_criteria)
    if smiles not in best_seen or fitness > best_seen[smiles][0]:
        best_seen[smiles] = (fitness, mol)

print(f"Générations effectuées : {evo_mol.nb_generations}")
print(f"Molécules uniques vues : {len(best_seen)}")

goal_names = [name for name, *_ in fitness_goals]
col_width   = 22

header_cols = "  ".join(f"{n:>{col_width}}" for n in goal_names)
print(f"\nTop {TOP_N} molécules finales :")
print(f"{'Rang':<5} {'Fitness':>10}  {header_cols}  SMILES")
print("-" * (5 + 12 + (col_width + 2) * len(goal_names) + 50))

top_items = sorted(best_seen.values(), key=lambda x: x[0], reverse=True)[:TOP_N]
for rank, (fitness, mol) in enumerate(top_items, start=1):
    smiles      = mol.get_representation(MolecularGraph).canonical_smiles
    detail_cols = "  ".join(f"{mol.value(n):>+{col_width}.4f}" for n in goal_names)
    print(f"{rank:<5} {fitness:>+10.4f}  {detail_cols}  {smiles}")

Générations effectuées : 17
Molécules uniques vues : 100

Top 30 molécules finales :
Rang     Fitness    SMILES
-------------------------------------------------------------------
1        +0.0000    C1=CC=CC=C1
2        +0.0000    NSC1=CC=CC=C1
3        +0.0000    FC1=CC=CC=C1
4        +0.0000    NC1=NC=CC=N1
5        +0.0000    OC1=CC=CC=C1
6        +0.0000    C=CC=CC=CC
7        +0.0000    C=CC=C(C)C=C
8        +0.0000    NC1=CC=CC=C1
9        +0.0000    C=CC=CC=C
10       +0.0000    SC1=CC=CC=C1
11       +0.0000    NC1=CC=CC=C1O
12       +0.0000    OC1=CC(F)=CC=C1
13       +0.0000    CC1=CC=CC=C1F
14       +0.0000    C1=CC=NC=C1
15       +0.0000    CC1=C(F)C=CC=C1
16       +0.0000    FC1=CC=NC=C1
17       +0.0000    CC1=CC=CC=C1N
18       +0.0000    CC1=CC=CC(F)=C1
19       +0.0000    NC1=C(S)C=CC=C1
20       +0.0000    NC1=CC=CC=N1
21       +0.0000    OC1=C(O)C=CC=C1
22       +0.0000    CC1=CC=CC(C)=N1
23       +0.0000    O=NC1=CC=CC=C1
24       +0.0000    O=C1C=CC=C1
25       +0.